# AI Job Agent — PHASE 4: Hard Filtering

**Goal:** take `retrieved_jobs.parquet` from Phase 3 (top-K jobs for the
**single live candidate**, PATH B) and apply deterministic, rule-based
filters — no embeddings, no LLM calls. This trims the candidate list down
to jobs that are *actually eligible*, before the more expensive detailed
matching in Phase 5.

**Single-candidate scope.** This notebook only implements **PATH B (live
candidate)**, matching what was used in Phase 2/3. There is no batch/PATH A
section here.

**Field mapping — verified against Phase 2's actual `candidate_profile.json`
schema** (`CANDIDATE_PREFERENCES` + the resume extraction fields), not
guessed:

```
retrieved_jobs.parquet (Phase 3)        candidate_profile.json (Phase 2)
         │                                          │
         └───────────────── hard filter funnel ─────┘
                    │
  country → city → work_arrangement → employment_type
                    → education → nationality → salary
                    │
              filtered_jobs.parquet  (→ Phase 5)
```

| Step | Job column (`jobs_prepared` / `retrieved_jobs`) | Candidate field (`candidate_profile.json`) |
|---|---|---|
| Country | `country` | `desired_countries` (skipped entirely if `willing_to_relocate` is `True`) |
| City | `city` | `desired_cities` (skipped entirely if `willing_to_relocate` is `True`) |
| Work arrangement | `work_arrangement`, `work_arrangement_inferred` | `work_arrangement_preference` — only enforced where `work_arrangement_inferred == False` |
| Employment type | `employment_type` | `employment_type_preference` |
| Education | `education_level` | `education_level` (a **list** of degrees held — the candidate's *highest* one is used) |
| Nationality | `nationality_requirement` | `nationality` |
| Salary | `salary_min`, `salary_max`, `salary_currency` | `salary_expectation_min/max/currency` |

Every filter is **optional**: a missing/empty/"Not Specified" candidate
value, or a missing job column, skips that step entirely rather than
dropping rows on data that isn't there.


---
## 1. Setup

In [1]:
import json
import shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)

WORK = Path("work")
OUT = Path("outputs")
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)


RUN_ID: 20260917_0949


---
## 2. Load Phase 3 output (retrieved jobs for the live candidate)

Upload `retrieved_jobs.parquet` from Phase 3's `outputs/live_candidate/`
folder (or the extracted `phase3_outputs.zip`).


In [2]:
from google.colab import files as _colab_files

print("Upload Phase 3's retrieved_jobs.parquet (PATH B / live candidate):")
uploaded_p3 = _colab_files.upload()

P3_DIR = WORK / "phase3"
P3_DIR.mkdir(exist_ok=True)
for name, content in uploaded_p3.items():
    (P3_DIR / name).write_bytes(content)

p3_file = P3_DIR / "retrieved_jobs.parquet"
assert p3_file.exists(), (
    "retrieved_jobs.parquet not found among the uploaded files — "
    "make sure you uploaded the PATH B output from Phase 3, not the batch one."
)
print("\nSaved:", p3_file.name)


Upload Phase 3's retrieved_jobs.parquet (PATH B / live candidate):


Saving retrieved_jobs.parquet to retrieved_jobs.parquet

Saved: retrieved_jobs.parquet


In [3]:
retrieved_df = pd.read_parquet(p3_file)

required_cols = {"rank", "similarity", "job_id"}
missing = required_cols - set(retrieved_df.columns)
assert not missing, f"retrieved_jobs.parquet is missing expected columns: {missing}"

print(f"Retrieved jobs (pre-filter): {len(retrieved_df):,}")
print(f"Similarity range: {retrieved_df['similarity'].min():.4f} – {retrieved_df['similarity'].max():.4f}")
retrieved_df.head(5)


Retrieved jobs (pre-filter): 150
Similarity range: 0.5575 – 0.6697


,rank,similarity,job_id,ats,slug,id,url_clean,title_clean,company_clean,jd_clean,...,salary_max,salary_currency,salary_period,responsibilities,industry,department,nationality_requirement,extraction_status,extraction_confidence,content_hash
0,1,0.669733,342910071506d6d0,dark,randstad.in,www.randstad.in/jobs/aiml-data-engineer_perungudi_d759fd34-8eb2-4ebf-aa2c-909ea4418be3/,https://www.randstad.in/jobs/aiml-data-engineer_perungudi_d759fd34-8eb2-4ebf-aa2c-909ea4418be3/,AI/ML Data Engineer,randstad.in,"* Programming & Foundations * Strong in Python, data structures, and algorithms. * Hands-on with...",...,NaN,None,Not Specified,[],None,None,None,ok,0.88,fc7fe53db983447c84c782af7021b9f8a88384bd
1,2,0.658302,396c91d624e77aca,dark,aijobs.com,www.aijobs.com/jobs/644193974-senior-lead-data-scientist-agentic-ai,https://www.aijobs.com/jobs/644193974-senior-lead-data-scientist-agentic-ai,Senior/Lead Data Scientist - Agentic AI,Tiger Analytics Inc.,Tiger Analytics is looking for experienced Data Scientists to join our fast-growing advanced ana...,...,NaN,None,Not Specified,"[Create AI agents and RAG databases, Develop AI agent tools for data retrieval and analysis, Pre...",Analytics consulting,Data Science,None,ok,0.98,1e255df8d0cad60ec453d803717b5a6b8c76d236
2,3,0.655182,2eac6bdbf5b755ed,dark,yulys.com,yulys.com/jobs/ai-data-engineer,https://yulys.com/jobs/ai-data-engineer,AI Data Engineer,Envision Technology Solutions,Role : AI Data EngineerLocation : Remote ( USA)Experience level – 15+ yrsJD :Proven experience a...,...,NaN,None,Not Specified,"[Design and build scalable production-grade software, Develop microservices and RESTful APIs, De...",Artificial Intelligence,Engineering,None,ok,0.97,c8767cd7641abb48e997987e2203ca84a666c18e
3,4,0.652827,0ee29f9b53ad16c3,dark,remotedxb.com,www.remotedxb.com/job/ai-engineer-codeninja--235803,https://www.remotedxb.com/job/ai-engineer-codeninja--235803,AI Engineer,CodeNinja,"Responsibilities Design, test, and optimize complex prompts for accurate financial text and data...",...,NaN,None,Not Specified,"[Design and optimize prompts for financial text extraction, Build analytics charts using generat...",Banking and Financial Services,AI Engineering,None,ok,0.95,31e437bafa5f182e43678061d258d455f369d6cb
4,5,0.643348,ecffa4de465c07d0,dark,snapjobsearch.com,snapjobsearch.com/jobs/view/10366516,https://snapjobsearch.com/jobs/view/10366516,Senior/Lead Data Scientist - Agentic AI,Tiger Analytics Inc.,Tiger Analytics is looking for experienced Data Scientists to join our fast-growing advanced ana...,...,NaN,None,Not Specified,"[Create AI agents and RAG databases to solve business problems, Develop AI agent tools for data ...",Advanced analytics consulting,Data Science and Artificial Intelligence,None,ok,0.98,8176a294b9c89760769cea51b87885562308b2c5


---
## 3. Load candidate profile (Phase 2)

Upload `candidate_profile.json` from Phase 2's `outputs/live_candidate/`
folder. All filter criteria in §4 are read straight from its known fields:
`desired_countries`, `desired_cities`, `work_arrangement_preference`,
`employment_type_preference`, `willing_to_relocate`, `nationality`,
`education_level` (a list of degrees held), and the `salary_expectation_*`
fields — exactly as written by Phase 2's `CANDIDATE_PREFERENCES` block and
its resume extraction.


In [4]:
print("Upload candidate_profile.json (from Phase 2):")
uploaded_profile = _colab_files.upload()

P2_DIR = WORK / "phase2"
P2_DIR.mkdir(exist_ok=True)
for name, content in uploaded_profile.items():
    (P2_DIR / name).write_bytes(content)

profile_path = P2_DIR / "candidate_profile.json"
assert profile_path.exists(), "candidate_profile.json not found among the uploaded files."

candidate_profile = json.loads(profile_path.read_text())
print(f"Loaded candidate profile: {candidate_profile.get('candidate_id')} "
      f"({candidate_profile.get('most_recent_title')})")


Upload candidate_profile.json (from Phase 2):


Saving candidate_profile.json to candidate_profile.json
Loaded candidate profile: e07ca2443daccbb5 (Freelance AI Engineer)


In [5]:
# Highest degree held. `education_level` in candidate_profile.json is a LIST
# (a candidate can hold several degrees) using the same EducationLevel enum
# as job postings. "Not Specified" entries are ignored when picking the max
# since they carry no ordinal information.
EDUCATION_HIERARCHY = ["High School", "Diploma", "Bachelor's Degree", "Master's Degree", "PhD"]
_edu_order = {level.lower(): i for i, level in enumerate(EDUCATION_HIERARCHY)}


def highest_education(levels: list) -> str:
    ranked = [lvl for lvl in (levels or []) if str(lvl).lower() in _edu_order]
    if not ranked:
        return None
    return max(ranked, key=lambda lvl: _edu_order[str(lvl).lower()])


candidate_highest_education = highest_education(candidate_profile.get("education_level"))

print("Fields pulled from candidate_profile.json:")
print(f"  desired_countries           : {candidate_profile.get('desired_countries')}")
print(f"  desired_cities              : {candidate_profile.get('desired_cities')}")
print(f"  willing_to_relocate         : {candidate_profile.get('willing_to_relocate')}")
print(f"  work_arrangement_preference : {candidate_profile.get('work_arrangement_preference')}")
print(f"  employment_type_preference  : {candidate_profile.get('employment_type_preference')}")
print(f"  nationality                 : {candidate_profile.get('nationality')}")
print(f"  education_level (raw list)  : {candidate_profile.get('education_level')}")
print(f"  -> highest degree held      : {candidate_highest_education}")
print(f"  salary_expectation_min/max  : {candidate_profile.get('salary_expectation_min')} / "
      f"{candidate_profile.get('salary_expectation_max')} {candidate_profile.get('salary_currency')} "
      f"per {candidate_profile.get('salary_period')}")


Fields pulled from candidate_profile.json:
  desired_countries           : ['Saudi Arabia']
  desired_cities              : []
  willing_to_relocate         : False
  work_arrangement_preference : Not Specified
  employment_type_preference  : Full-time
  nationality                 : Saudi
  education_level (raw list)  : ['Diploma', "Bachelor's Degree"]
  -> highest degree held      : Bachelor's Degree
  salary_expectation_min/max  : None / None SAR per month


---
## 4. Configuration — hard filter criteria

Everything defaults straight from `candidate_profile.json` (§3) — nothing
here is guessed. Override any value by hand if you want to test the funnel
with different assumptions than what's in the profile.

A value of `None` / `[]` / `"Not Specified"` skips that filter step
entirely — no rows get dropped for that reason.


In [6]:
def _is_unset(value) -> bool:
    return value in (None, "", [], "Not Specified")


CONFIG = {
    # Country/city preference. Both are skipped entirely if the candidate
    # is willing to relocate — a relocating candidate isn't limited to
    # their stated preference.
    "allowed_countries": None if candidate_profile.get("willing_to_relocate") else candidate_profile.get("desired_countries"),
    "allowed_cities": None if candidate_profile.get("willing_to_relocate") else candidate_profile.get("desired_cities"),
    "enforce_city_for_remote": False,  # a remote job's city rarely matters to the candidate

    # Work arrangement, e.g. "Remote" / "Hybrid" / "Onsite" / "Not Specified".
    # Only enforced on rows where work_arrangement_inferred == False (see §6).
    "accepted_work_arrangements": (
        None if _is_unset(candidate_profile.get("work_arrangement_preference"))
        else [candidate_profile["work_arrangement_preference"]]
    ),

    # Employment type, e.g. "Full-time" / "Part-time" / "Contract" / etc.
    "accepted_employment_types": (
        None if _is_unset(candidate_profile.get("employment_type_preference"))
        else [candidate_profile["employment_type_preference"]]
    ),

    # Highest degree the candidate holds (derived in §3 from the
    # education_level list). A job is kept if its required level is at or
    # below this.
    "candidate_education_level": candidate_highest_education,
    "education_hierarchy": EDUCATION_HIERARCHY,

    "candidate_nationality": candidate_profile.get("nationality"),
    # If True, a job with an unreadable/unparseable nationality_requirement
    # value is dropped instead of kept. Keeping (False) is the safer default
    # since we'd rather over-include than silently lose an eligible job.
    "strict_nationality_check": False,

    # Salary expectation. A job is kept if the ranges overlap, the job has
    # no stated salary, or currencies don't match (we can't compare them
    # without a conversion rate, so we don't penalize the job for it).
    "salary_expectation_min": candidate_profile.get("salary_expectation_min"),
    "salary_expectation_max": candidate_profile.get("salary_expectation_max"),
    "salary_currency": candidate_profile.get("salary_currency"),

    # Minimum number of surviving jobs below which we print a warning
    # suggesting the filters may be too strict.
    "min_survivors_warning": 10,
}

print("CONFIG:", json.dumps(CONFIG, indent=2, default=str))


CONFIG: {
  "allowed_countries": [
    "Saudi Arabia"
  ],
  "allowed_cities": [],
  "enforce_city_for_remote": false,
  "accepted_work_arrangements": null,
  "accepted_employment_types": [
    "Full-time"
  ],
  "candidate_education_level": "Bachelor's Degree",
  "education_hierarchy": [
    "High School",
    "Diploma",
    "Bachelor's Degree",
    "Master's Degree",
    "PhD"
  ],
  "candidate_nationality": "Saudi",
  "strict_nationality_check": false,
  "salary_expectation_min": null,
  "salary_expectation_max": null,
  "salary_currency": "SAR",
  "min_survivors_warning": 10
}


---
## 5. Filter functions

Each function returns a **boolean keep-mask** the same length as its input
`DataFrame`, and prints a note when the job data doesn't have the column it
needs (in which case it keeps everything rather than guessing). None of
these mutate `retrieved_df` directly — the funnel in §6 applies them one at
a time so we can report how many jobs each step removes.


In [7]:
def _missing_column(df: pd.DataFrame, col: str) -> bool:
    if col not in df.columns:
        print(f"  [NOTE] '{col}' not found in retrieved_jobs.parquet — skipping this filter.")
        return True
    return False


def _cell_values(value) -> list:
    '''Normalize a single job-column cell into a list of non-null values.
    Some Phase 1 fields are scalars (e.g. "Bachelor's Degree") and some are
    lists (e.g. several accepted degrees, or several accepted nationalities)
    — this handles both the same way, which is also what keeps pd.isna()
    from blowing up on a list-valued cell ("truth value of an array with
    more than one element is ambiguous").'''
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        items = list(value)
    else:
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass
        items = [value]
    return [v for v in items if v is not None and not (isinstance(v, float) and pd.isna(v))]


def mask_country(df: pd.DataFrame, allowed: list) -> np.ndarray:
    '''Keep jobs whose country is in `allowed` (case-insensitive).
    Rows with a missing country are dropped, since we cannot verify them.'''
    if not allowed or _missing_column(df, "country"):
        return np.ones(len(df), dtype=bool)
    allowed_lower = {str(c).strip().lower() for c in allowed}

    def check(value):
        values = _cell_values(value)
        if not values:
            return False
        return any(str(v).strip().lower() in allowed_lower for v in values)

    return df["country"].apply(check).to_numpy()


def mask_city(df: pd.DataFrame, allowed: list, enforce_for_remote: bool) -> np.ndarray:
    '''Keep jobs whose city is in `allowed` (case-insensitive). Remote jobs
    are exempted by default, since the city of a remote posting is usually
    irrelevant to the candidate.'''
    if not allowed or _missing_column(df, "city"):
        return np.ones(len(df), dtype=bool)
    allowed_lower = {str(c).strip().lower() for c in allowed}

    def check(value):
        values = _cell_values(value)
        return any(str(v).strip().lower() in allowed_lower for v in values)

    keep = df["city"].apply(check).to_numpy()

    if not enforce_for_remote and "work_arrangement" in df.columns:
        def is_remote(value):
            values = _cell_values(value)
            return any(str(v).strip().lower() == "remote" for v in values)
        keep = keep | df["work_arrangement"].apply(is_remote).to_numpy()

    return keep


def mask_work_arrangement(df: pd.DataFrame, accepted: list) -> np.ndarray:
    '''Keep jobs whose work_arrangement is in `accepted` (case-insensitive).
    Only enforced on rows where work_arrangement_inferred == False — an
    inferred value is a guess, not a stated fact, so we give those jobs the
    benefit of the doubt rather than dropping them on uncertain data.'''
    if not accepted or _missing_column(df, "work_arrangement"):
        return np.ones(len(df), dtype=bool)
    accepted_lower = {str(a).strip().lower() for a in accepted}

    def check(value):
        values = _cell_values(value)
        return any(str(v).strip().lower() in accepted_lower for v in values)

    matches = df["work_arrangement"].apply(check).to_numpy()

    if "work_arrangement_inferred" in df.columns:
        inferred = df["work_arrangement_inferred"].fillna(False).astype(bool).to_numpy()
        # inferred rows are always kept; stated rows must match
        return matches | inferred

    return matches


def mask_employment_type(df: pd.DataFrame, accepted: list) -> np.ndarray:
    '''Keep jobs whose employment_type is in `accepted` (case-insensitive).'''
    if not accepted or _missing_column(df, "employment_type"):
        return np.ones(len(df), dtype=bool)
    accepted_lower = {str(a).strip().lower() for a in accepted}

    def check(value):
        values = _cell_values(value)
        return any(str(v).strip().lower() in accepted_lower for v in values)

    return df["employment_type"].apply(check).to_numpy()


def mask_education(df: pd.DataFrame, candidate_level: str, hierarchy: list) -> np.ndarray:
    '''Keep jobs whose required education level is at or below the
    candidate's highest held level in `hierarchy`. A job cell can list
    several acceptable degrees (kept if the candidate meets ANY of them) or
    be missing/unrecognized (kept — that isn't proof of ineligibility).'''
    if not candidate_level or _missing_column(df, "education_level"):
        return np.ones(len(df), dtype=bool)

    order = {level.strip().lower(): i for i, level in enumerate(hierarchy)}
    cand_idx = order.get(str(candidate_level).strip().lower())
    if cand_idx is None:
        print(f"  [WARNING] candidate_education_level '{candidate_level}' not found in "
              f"education_hierarchy — skipping education filter entirely.")
        return np.ones(len(df), dtype=bool)

    def check(value):
        values = _cell_values(value)
        if not values:
            return True  # no stated requirement -> keep
        recognized = [order.get(str(v).strip().lower()) for v in values]
        recognized = [i for i in recognized if i is not None]
        if not recognized:
            return True  # nothing recognizable (e.g. all "Not Specified") -> keep
        return any(idx <= cand_idx for idx in recognized)

    return df["education_level"].apply(check).to_numpy()


def mask_nationality(df: pd.DataFrame, candidate_nationality: str, strict: bool) -> np.ndarray:
    '''Keep jobs that are open to all nationalities, or that explicitly
    list the candidate's nationality as accepted. A job cell can list
    several accepted nationalities. Jobs with no candidate nationality to
    check against are kept unless `strict` is True.'''
    if _missing_column(df, "nationality_requirement"):
        return np.ones(len(df), dtype=bool)

    open_values = {"", "none", "any", "open", "all", "no restriction", "nan", "not specified"}

    def check(value):
        values = _cell_values(value)
        normalized = [str(v).strip().lower() for v in values]
        if not normalized or any(v in open_values for v in normalized):
            return True  # open to everyone
        if not candidate_nationality:
            return not strict  # unknown candidate nationality: keep unless strict
        cand = str(candidate_nationality).strip().lower()
        return any(cand in v for v in normalized)

    return df["nationality_requirement"].apply(check).to_numpy()


def mask_salary(df: pd.DataFrame, cand_min, cand_max, cand_currency: str) -> np.ndarray:
    '''Keep jobs whose salary range overlaps the candidate's expectation.
    A job with no stated salary, or a currency that doesn't match the
    candidate's (we can't convert without a rate), is kept rather than
    dropped — we only filter on a comparison we can actually make.'''
    needed_cols = {"salary_min", "salary_max", "salary_currency"}
    if (cand_min is None and cand_max is None) or not needed_cols.issubset(df.columns):
        if not needed_cols.issubset(df.columns):
            print(f"  [NOTE] salary columns {sorted(needed_cols - set(df.columns))} not found "
                  f"in retrieved_jobs.parquet — skipping this filter.")
        return np.ones(len(df), dtype=bool)

    def scalar(value):
        values = _cell_values(value)
        return values[0] if values else None

    def check(row):
        job_min = scalar(row["salary_min"])
        job_max = scalar(row["salary_max"])
        job_currency = scalar(row["salary_currency"])
        if job_min is None and job_max is None:
            return True  # no stated salary -> keep
        if cand_currency and job_currency is not None and str(job_currency).strip().lower() != str(cand_currency).strip().lower():
            return True  # different currency, can't compare -> keep
        lo = cand_min if cand_min is not None else float("-inf")
        hi = cand_max if cand_max is not None else float("inf")
        job_lo = job_min if job_min is not None else float("-inf")
        job_hi = job_max if job_max is not None else float("inf")
        return job_lo <= hi and job_hi >= lo  # ranges overlap

    return df.apply(check, axis=1).to_numpy()


---
## 6. Run the filtering funnel

Filters are applied **sequentially** so each step's impact is visible on
top of the previous ones — this is easier to debug than one combined
boolean mask when a step turns out to be too aggressive.


In [8]:
funnel_records = []
current_df = retrieved_df.copy()

def apply_step(df: pd.DataFrame, name: str, mask: np.ndarray) -> pd.DataFrame:
    before = len(df)
    survivors = df[mask]
    after = len(survivors)
    funnel_records.append({
        "step": name,
        "before": before,
        "after": after,
        "dropped": before - after,
    })
    print(f"  {name:20s}: {before:5d} -> {after:5d}  (-{before - after})")
    return survivors


print("Filter funnel (single candidate):\n")

current_df = apply_step(
    current_df, "country",
    mask_country(current_df, CONFIG["allowed_countries"]),
)
current_df = apply_step(
    current_df, "city",
    mask_city(current_df, CONFIG["allowed_cities"], CONFIG["enforce_city_for_remote"]),
)
current_df = apply_step(
    current_df, "work_arrangement",
    mask_work_arrangement(current_df, CONFIG["accepted_work_arrangements"]),
)
current_df = apply_step(
    current_df, "employment_type",
    mask_employment_type(current_df, CONFIG["accepted_employment_types"]),
)
current_df = apply_step(
    current_df, "education",
    mask_education(current_df, CONFIG["candidate_education_level"], CONFIG["education_hierarchy"]),
)
current_df = apply_step(
    current_df, "nationality",
    mask_nationality(current_df, CONFIG["candidate_nationality"], CONFIG["strict_nationality_check"]),
)
current_df = apply_step(
    current_df, "salary",
    mask_salary(current_df, CONFIG["salary_expectation_min"], CONFIG["salary_expectation_max"], CONFIG["salary_currency"]),
)

filtered_df = current_df.reset_index(drop=True)
filtered_df["post_filter_rank"] = range(1, len(filtered_df) + 1)

print(f"\nFinal: {len(filtered_df):,} / {len(retrieved_df):,} jobs survived all filters "
      f"({len(filtered_df) / max(len(retrieved_df), 1):.1%})")


Filter funnel (single candidate):

  country             :   150 ->     4  (-146)
  city                :     4 ->     4  (-0)
  work_arrangement    :     4 ->     4  (-0)
  employment_type     :     4 ->     1  (-3)
  education           :     1 ->     1  (-0)
  nationality         :     1 ->     1  (-0)
  salary              :     1 ->     1  (-0)

Final: 1 / 150 jobs survived all filters (0.7%)


### 6.1 Funnel summary

In [9]:
funnel_df = pd.DataFrame(funnel_records)
funnel_df["dropped_pct"] = (funnel_df["dropped"] / funnel_df["before"]).round(4)
funnel_df


,step,before,after,dropped,dropped_pct
0,country,150,4,146,0.9733
1,city,4,4,0,0.0000
2,work_arrangement,4,4,0,0.0000
3,employment_type,4,1,3,0.7500
4,education,1,1,0,0.0000
5,nationality,1,1,0,0.0000
6,salary,1,1,0,0.0000


---
## 7. Review filtered results


In [10]:
display_cols = ["post_filter_rank", "rank", "similarity", "title_clean", "company_clean",
                "city", "country", "work_arrangement", "employment_type",
                "education_level", "nationality_requirement"]
display_cols = [c for c in display_cols if c in filtered_df.columns]

filtered_df[display_cols].head(20).style.format({"similarity": "{:.4f}"})


,post_filter_rank,rank,similarity,title_clean,company_clean,city,country,work_arrangement,employment_type,education_level,nationality_requirement
0,1,54,0.5814,Big Data and AI Manager,Confidential,Riyadh,Saudi Arabia,On-site,Full-time,"[""Bachelor's Degree"" ""Master's Degree"" 'PhD']",None


### 7.1 Diagnostics

In [11]:
if len(filtered_df) == 0:
    print("[WARNING] Every job was filtered out. Double-check CONFIG — in "
          "particular allowed_countries/allowed_cities, which are the two "
          "most common causes of an empty result.")
elif len(filtered_df) < CONFIG["min_survivors_warning"]:
    print(f"[WARNING] Only {len(filtered_df)} jobs survived filtering "
          f"(below min_survivors_warning={CONFIG['min_survivors_warning']}). "
          f"Consider relaxing CONFIG or increasing Phase 3's top_k so there's "
          f"a larger pool to filter from.")
else:
    print(f"{len(filtered_df)} jobs survived filtering — looks like a healthy pool for Phase 5.")

# Which single step removed the most jobs, for a quick "what's my tightest constraint" read
if len(funnel_df) and funnel_df["dropped"].sum() > 0:
    tightest = funnel_df.loc[funnel_df["dropped"].idxmax()]
    print(f"\nTightest constraint: '{tightest['step']}' dropped {int(tightest['dropped'])} jobs "
          f"({tightest['dropped_pct']:.1%} of what entered that step).")


[WARNING] Only 1 jobs survived filtering (below min_survivors_warning=10). Consider relaxing CONFIG or increasing Phase 3's top_k so there's a larger pool to filter from.

Tightest constraint: 'country' dropped 146 jobs (97.3% of what entered that step).


---
## 8. Save output


In [12]:
LIVE_OUT = OUT / "live_candidate"
LIVE_OUT.mkdir(parents=True, exist_ok=True)

filtered_df.to_parquet(LIVE_OUT / "filtered_jobs.parquet", index=False)
funnel_df.to_csv(LIVE_OUT / "filter_funnel.csv", index=False)

print(f"Saved {LIVE_OUT / 'filtered_jobs.parquet'}  ({len(filtered_df)} rows)")
print(f"Saved {LIVE_OUT / 'filter_funnel.csv'}  ({len(funnel_df)} rows)")


Saved outputs/live_candidate/filtered_jobs.parquet  (1 rows)
Saved outputs/live_candidate/filter_funnel.csv  (7 rows)


---
## 9. Data dictionary


In [13]:
DATA_DICTIONARY = '''# Phase 4 outputs — Data Dictionary

## live_candidate/filtered_jobs.parquet

Same columns as Phase 3's retrieved_jobs.parquet, with one addition:

| Column | Notes |
|---|---|
| post_filter_rank | 1 = most similar job that survived filtering. Contiguous, no gaps. Distinct from `rank`, which is the original Phase 3 rank before any filtering. |
| rank, similarity | Carried over from Phase 3 unchanged. |
| job_id | Foreign key into jobs_prepared.parquet from Phase 1. |

This file is the direct input to PHASE 5 (Detailed Matching) — no need to
re-run retrieval or re-apply these filters.

## live_candidate/filter_funnel.csv

One row per filter step, in the order applied: `step, before, after,
dropped, dropped_pct`. Useful for explaining to a candidate (or debugging
CONFIG) why the final pool looks the way it does.

## Filtering contract
- Filters are deterministic and rule-based — no embeddings or LLM calls.
- Every CONFIG value is read directly from candidate_profile.json (Phase 2);
  a None/empty/"Not Specified" value skips that filter entirely rather than
  dropping rows on missing data.
- country/city are skipped entirely when the candidate's willing_to_relocate
  is True.
- work_arrangement is only enforced on rows where work_arrangement_inferred
  == False, so a guessed value never costs a job its spot.
- education compares the candidate's *highest* held degree (from the
  education_level list) against each job's required level.
- Rows with unreadable/unrecognized education, nationality, or salary
  currency values are kept by default rather than dropped.
'''

(OUT / "PHASE4_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")


Data dictionary written.


---
## 10. Final verification


In [14]:
filtered_path = OUT / "live_candidate" / "filtered_jobs.parquet"
reloaded = pd.read_parquet(filtered_path)

assert reloaded["job_id"].is_unique, "duplicate job_id in filtered output"
assert reloaded["post_filter_rank"].min() == 1 if len(reloaded) else True
assert reloaded["post_filter_rank"].is_monotonic_increasing

print(f"rows       : {len(reloaded)}")
if len(reloaded):
    print(f"top match  : {reloaded.iloc[0]['title_clean']} "
          f"(similarity={reloaded.iloc[0]['similarity']:.4f})")
print("STATUS: ready for Phase 5")


rows       : 1
top match  : Big Data and AI Manager (similarity=0.5814)
STATUS: ready for Phase 5


---
## 11. Persist outputs


In [15]:
zip_path = shutil.make_archive("phase4_outputs", "zip", OUT)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.2f} MB)")

_colab_files.download(zip_path)


Created /content/phase4_outputs.zip (0.01 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Next step — PHASE 5

`filtered_jobs.parquet` has the same job columns as Phase 3's output, now
narrowed down to jobs the candidate is actually eligible for. Phase 5
(Detailed Matching) works directly off this file — score/rank the survivors
on finer-grained criteria (skills overlap, seniority fit, etc.) that go
beyond what a hard yes/no filter can express.
